# 🚀 Interactive Customer Churn Dashboard
**Future Interns — Task 2 | Customer Retention & Churn Analysis**

---

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../dataset/telecom_churn_data.csv')
df['JoinDate'] = pd.to_datetime(df['JoinDate'])
df['Churn_Label'] = df['Churn'].map({1: 'Churned', 0: 'Active'})
df['Tenure_Group'] = pd.cut(df['Tenure_Months'], bins=[0,6,12,24,36,72],
    labels=['0-6 mo','7-12 mo','13-24 mo','25-36 mo','36+ mo'])

COLORS = {'Active': '#2ecc71', 'Churned': '#e74c3c'}
print('✅ Data loaded:', df.shape)

✅ Data loaded: (1500, 20)


## 📊 KPI Dashboard

In [ ]:
total = len(df)
churned = df['Churn'].sum()
active = total - churned
churn_rate = churned/total*100
avg_tenure = df['Tenure_Months'].mean()
avg_monthly = df['MonthlyCharges'].mean()
revenue_lost = df[df['Churn']==1]['TotalCharges'].sum()

fig = make_subplots(rows=1, cols=6,
    specs=[[{'type':'indicator'}]*6],
    column_widths=[1,1,1,1,1,1])

kpis = [
    ('Total Customers', total, None, '#3498db'),
    ('Active Customers', active, None, '#2ecc71'),
    ('Churned Customers', churned, None, '#e74c3c'),
    ('Churn Rate', churn_rate, '%', '#e67e22'),
    ('Avg Tenure (mo)', avg_tenure, None, '#9b59b6'),
    ('Avg Monthly (₹)', avg_monthly, None, '#1abc9c'),
]
for i, (title, val, suffix, color) in enumerate(kpis, 1):
    num_fmt = f'.1f' if isinstance(val, float) else ','
    fig.add_trace(go.Indicator(
        mode='number',
        value=val,
        title={'text': f'<b>{title}</b>', 'font': {'size':13}},
        number={'font': {'size':28, 'color': color}, 'valueformat': num_fmt,
                'suffix': suffix if suffix else ''},
    ), row=1, col=i)

fig.update_layout(height=160, title_text='📊 Key Performance Indicators',
    title_font_size=16, paper_bgcolor='#f8f9fa', margin=dict(t=50,b=10,l=10,r=10))
fig.show()

## 🍩 Churn Distribution + Contract Type

In [ ]:
fig = make_subplots(rows=1, cols=2,
    specs=[[{'type':'pie'}, {'type':'bar'}]],
    subplot_titles=['Overall Churn Distribution', 'Churn Rate by Contract Type'])

churn_counts = df['Churn_Label'].value_counts()
fig.add_trace(go.Pie(
    labels=churn_counts.index, values=churn_counts.values,
    hole=0.45, marker_colors=['#2ecc71','#e74c3c'],
    textinfo='label+percent', textfont_size=12,
    pull=[0, 0.05]), row=1, col=1)

ct = df.groupby('Contract')['Churn'].mean().mul(100).round(1).reset_index()
ct.columns = ['Contract', 'ChurnRate']
ct = ct.sort_values('ChurnRate', ascending=False)
fig.add_trace(go.Bar(
    x=ct['Contract'], y=ct['ChurnRate'],
    marker_color=['#e74c3c','#f39c12','#2ecc71'],
    text=ct['ChurnRate'].apply(lambda x: f'{x:.1f}%'),
    textposition='outside', showlegend=False), row=1, col=2)

fig.update_yaxes(title_text='Churn Rate (%)', row=1, col=2)
fig.update_layout(height=400, title_text='🍩 Churn Overview',
    title_font_size=16, showlegend=True, legend=dict(orientation='h', y=-0.1))
fig.show()

## 📈 Churn by Internet Service & Payment Method

In [ ]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Churn Rate by Internet Service', 'Churn Rate by Payment Method'])

is_ = df.groupby('InternetService')['Churn'].mean().mul(100).round(1).reset_index()
is_.columns = ['Service','ChurnRate']
is_ = is_.sort_values('ChurnRate', ascending=False)
fig.add_trace(go.Bar(x=is_['Service'], y=is_['ChurnRate'],
    marker_color=['#e74c3c','#3498db','#2ecc71'],
    text=is_['ChurnRate'].apply(lambda x: f'{x:.1f}%'),
    textposition='outside', showlegend=False), row=1, col=1)

pm = df.groupby('PaymentMethod')['Churn'].mean().mul(100).round(1).reset_index()
pm.columns = ['Method','ChurnRate']
pm = pm.sort_values('ChurnRate', ascending=False)
fig.add_trace(go.Bar(x=pm['Method'], y=pm['ChurnRate'],
    marker_color='#e67e22',
    text=pm['ChurnRate'].apply(lambda x: f'{x:.1f}%'),
    textposition='outside', showlegend=False), row=1, col=2)

fig.update_yaxes(title_text='Churn Rate (%)')
fig.update_layout(height=420, title_text='📡 Service & Payment Analysis', title_font_size=16)
fig.show()

## ⏱️ Tenure Group & Churn Reasons

In [ ]:
fig = make_subplots(rows=1, cols=2,
    specs=[[{'type':'bar'}, {'type':'pie'}]],
    subplot_titles=['Churn Rate by Tenure Group', 'Churn Reasons Breakdown'])

tg = df.groupby('Tenure_Group', observed=True)['Churn'].mean().mul(100).round(1).reset_index()
tg.columns = ['Group','ChurnRate']
fig.add_trace(go.Bar(x=tg['Group'], y=tg['ChurnRate'],
    marker_color=['#e74c3c','#e67e22','#f39c12','#3498db','#2ecc71'],
    text=tg['ChurnRate'].apply(lambda x: f'{x:.1f}%'),
    textposition='outside', showlegend=False), row=1, col=1)

reasons = df[df['Churn']==1]['ChurnReason'].value_counts().reset_index()
reasons.columns = ['Reason','Count']
fig.add_trace(go.Pie(
    labels=reasons['Reason'], values=reasons['Count'],
    hole=0.4, textinfo='label+percent',
    marker_colors=px.colors.qualitative.Set2), row=1, col=2)

fig.update_yaxes(title_text='Churn Rate (%)', row=1, col=1)
fig.update_layout(height=420, title_text='⏱️ Tenure & Churn Reasons', title_font_size=16, showlegend=False)
fig.show()

## 📅 Cohort Retention Trend

In [ ]:
cohort = df.groupby('CohortMonth').agg(Total=('Churn','count'),Churned=('Churn','sum')).reset_index()
cohort['RetentionRate'] = ((cohort['Total']-cohort['Churned'])/cohort['Total']*100).round(1)
cohort['ChurnRate'] = (cohort['Churned']/cohort['Total']*100).round(1)
cohort = cohort.sort_values('CohortMonth')

fig = go.Figure()
fig.add_trace(go.Scatter(x=cohort['CohortMonth'], y=cohort['RetentionRate'],
    mode='lines+markers', name='Retention Rate %',
    line=dict(color='#2ecc71', width=3), marker=dict(size=8),
    fill='tozeroy', fillcolor='rgba(46,204,113,0.1)'))
fig.add_trace(go.Scatter(x=cohort['CohortMonth'], y=cohort['ChurnRate'],
    mode='lines+markers', name='Churn Rate %',
    line=dict(color='#e74c3c', width=3), marker=dict(size=8)))
fig.add_hline(y=cohort['RetentionRate'].mean(), line_dash='dash',
    line_color='#2ecc71', annotation_text=f'Avg Retention: {cohort["RetentionRate"].mean():.1f}%')
fig.update_layout(height=400, title='📅 Monthly Cohort — Retention vs Churn Trend',
    title_font_size=16, xaxis_title='Join Month', yaxis_title='Rate (%)',
    legend=dict(orientation='h', y=1.12), xaxis_tickangle=-45)
fig.show()

## 🔵 Scatter: Tenure vs Monthly Charges

In [ ]:
sample = df.sample(700, random_state=42)
fig = px.scatter(sample, x='Tenure_Months', y='MonthlyCharges',
    color='Churn_Label', color_discrete_map=COLORS,
    hover_data=['CustomerID','Contract','InternetService'],
    opacity=0.7, title='🔵 Tenure vs Monthly Charges (by Churn Status)',
    labels={'Tenure_Months':'Tenure (Months)','MonthlyCharges':'Monthly Charges (₹)','Churn_Label':'Status'})
fig.update_traces(marker=dict(size=6))
fig.update_layout(height=450, title_font_size=16, legend_title='Status')
fig.show()

## 🗺️ Churn by Region

In [ ]:
region_churn = df.groupby('Region').agg(
    Total=('Churn','count'), Churned=('Churn','sum')).reset_index()
region_churn['ChurnRate'] = (region_churn['Churned']/region_churn['Total']*100).round(1)

fig = px.bar(region_churn, x='Region', y='ChurnRate',
    color='ChurnRate', color_continuous_scale='RdYlGn_r',
    text='ChurnRate', title='🗺️ Churn Rate by Region',
    labels={'ChurnRate':'Churn Rate (%)'})
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=380, title_font_size=16, coloraxis_showscale=False)
fig.show()

---
## 💡 Business Recommendations

| Priority | Recommendation |
|----------|---------------|
| 🔴 High | Convert Month-to-Month customers to annual plans with discounts |
| 🔴 High | Target new customers (0–6 months) with onboarding support |
| 🟡 Medium | Investigate Fiber Optic service quality issues |
| 🟡 Medium | Incentivize auto-pay adoption over electronic check |
| 🟢 Low | Launch loyalty rewards for 24+ month customers |
| 🟢 Low | Cross-sell products to single-product customers |